# 01 — Сравнение forecasting-моделей

## Цель

На единой frozen-панели из `00_data_final.ipynb` строится честный
walk-forward benchmark моделей прогнозирования муниципальных расходов.

На этом этапе сравниваются:

1. **Prophet** — обязательный конкурсный ориентир;
2. **Seasonal Growth** — простой сезонный baseline;
3. **Factor Anchor** — панельный baseline;
4. **AutoETS** — классическая статистическая модель;
5. **Amazon Chronos-Bolt-base** — современная foundation model временных рядов;
6. **Reference forecasting ensemble** — простое среднее трёх
   методологически различных экспертов.

Главная метрика — **MAE**, как требует конкурс.

`R²` рассчитывается дополнительно только как диагностическая метрика.

## Методологическое правило

Для каждого forecast origin любая модель получает только историю:

`period <= origin`

Будущие значения используются исключительно после построения прогноза —
для расчёта метрик.

Все методы сравниваются на одинаковых:

- 2 016 муниципальных рядах;
- forecast origins;
- горизонтах `h = 1, 2, 3`;
- target months;
- метрике MAE.

Это позволяет напрямую и корректно сравнить итоговый подход с Prophet.

In [ ]:
from pathlib import Path
import json
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from chronos import BaseChronosPipeline
from prophet import Prophet
from statsforecast import StatsForecast
from statsforecast.models import AutoETS
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()

    for candidate in [cwd, cwd.parent]:
        if (
            (candidate / "data").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Не удалось определить корень проекта. "
        "Запускайте notebook из корня проекта или каталога notebooks/."
    )


PROJECT_ROOT = find_project_root()
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

WIDE_FILE = (
    INTERIM_DIR
    / "consumption_total_wide_2016x24.parquet"
)

PANEL_FILE = (
    INTERIM_DIR
    / "consumption_total_panel_2016.parquet"
)

DATA_MANIFEST_FILE = (
    INTERIM_DIR
    / "data_final_manifest.json"
)

for path in [
    WIDE_FILE,
    PANEL_FILE,
    DATA_MANIFEST_FILE,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Не найден обязательный входной артефакт: {path}"
        )

with DATA_MANIFEST_FILE.open(
    "r",
    encoding="utf-8",
) as f:
    data_manifest = json.load(f)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Input artifacts: OK")
print(
    "Source SHA256:",
    data_manifest["source"]["sha256"],
)


## 1. Frozen forecasting panel

Используется только результат `00_data_final.ipynb`.

На этом этапе исходный CSV больше не реконструируется:
все модели работают на одной и той же frozen-панели.

In [ ]:
wide_export = pd.read_parquet(
    WIDE_FILE
)

assert "series_id" in wide_export.columns

wide_ref = (
    wide_export
    .set_index("series_id")
    .copy()
)

wide_ref.columns = pd.to_datetime(
    wide_ref.columns
)

wide_ref = wide_ref.sort_index(
    axis=1
)

panel = pd.read_parquet(
    PANEL_FILE
)

panel["period"] = pd.to_datetime(
    panel["period"]
)

print("wide_ref:", wide_ref.shape)
print("panel:", panel.shape)
print(
    "period:",
    wide_ref.columns.min(),
    "—",
    wide_ref.columns.max(),
)

assert wide_ref.shape == (2016, 24)
assert panel.shape[0] == 48_384
assert panel["series_id"].nunique() == 2016
assert panel["period"].nunique() == 24
assert wide_ref.isna().sum().sum() == 0
assert panel.isna().sum().sum() == 0
assert not panel.duplicated(
    ["series_id", "period"]
).any()

print("Panel integrity: PASSED")


## 2. Единый multi-horizon evaluation protocol

Оцениваются четыре последовательных forecast origin:

- 2024-06-01;
- 2024-07-01;
- 2024-08-01;
- 2024-09-01.

Для каждого origin строятся прогнозы на:

`h = 1, 2, 3`

Итого для каждой модели:

`2 016 series × 4 origins × 3 horizons = 24 192 forecasts`

Jun–Sep используется как development/challenger benchmark проекта
и не называется новым untouched holdout.

In [ ]:
TEST_ORIGINS = pd.to_datetime([
    "2024-06-01",
    "2024-07-01",
    "2024-08-01",
    "2024-09-01",
])

HORIZONS = [
    1,
    2,
    3,
]

N_SERIES = len(
    wide_ref
)

N_FORECASTS_PER_MODEL = (
    N_SERIES
    * len(TEST_ORIGINS)
    * len(HORIZONS)
)

eval_rows = []

for origin in TEST_ORIGINS:
    for h in HORIZONS:
        target = (
            origin
            + pd.DateOffset(
                months=h
            )
        )

        eval_rows.append(
            pd.DataFrame({
                "series_id":
                    wide_ref.index,

                "origin":
                    origin,

                "h":
                    h,

                "target_period":
                    target,

                "actual_value":
                    wide_ref[
                        target
                    ].to_numpy(),
            })
        )

eval_grid = pd.concat(
    eval_rows,
    ignore_index=True,
)

print(
    "Rows:",
    len(eval_grid),
)

print(
    "Series:",
    eval_grid[
        "series_id"
    ].nunique(),
)

print(
    "Origins:",
    eval_grid[
        "origin"
    ].nunique(),
)

print(
    "Horizons:",
    sorted(
        eval_grid[
            "h"
        ].unique()
    ),
)

assert len(eval_grid) == 24_192
assert N_FORECASTS_PER_MODEL == 24_192
assert eval_grid["series_id"].nunique() == 2016
assert eval_grid["origin"].nunique() == 4
assert set(eval_grid["h"]) == {1, 2, 3}
assert not eval_grid.duplicated(
    [
        "series_id",
        "origin",
        "h",
    ]
).any()
assert eval_grid.isna().sum().sum() == 0

print("Evaluation protocol: PASSED")


## 3. Forecasting methods

### 3.1 Панельное разложение

Для каждого месяца рассчитывается общий фактор панели:

`f(t) = median_i(log(y(i,t)))`

Относительный ряд муниципалитета:

`r(i,t) = log(y(i,t)) - f(t)`

Такое представление отделяет общую динамику расходов
от локального поведения конкретного муниципального ряда.

In [ ]:
def panel_log_factor(
    wide: pd.DataFrame,
) -> pd.Series:
    return np.log(
        wide
    ).median(
        axis=0
    )


def seasonal_growth_forecast(
    wide: pd.DataFrame,
    horizons,
    growth_window: int = 3,
) -> pd.DataFrame:
    horizons = list(
        horizons
    )

    log_level = np.log(
        wide
    )

    log_yoy = (
        log_level
        - log_level.shift(
            12,
            axis=1,
        )
    )

    recent_growth = (
        log_yoy
        .iloc[
            :,
            -growth_window:
        ]
        .mean(
            axis=1
        )
    )

    result = {}

    for h in horizons:
        seasonal_anchor = (
            wide.iloc[
                :,
                -12 + h - 1
            ]
        )

        result[h] = (
            seasonal_anchor
            * np.exp(
                recent_growth
            )
        )

    return pd.DataFrame(
        result,
        index=wide.index,
    )


def factor_future(
    factor: pd.Series,
    horizons,
    growth_window: int = 1,
) -> pd.Series:
    factor_level = pd.DataFrame(
        [
            np.exp(
                factor.to_numpy()
            )
        ],
        columns=factor.index,
    )

    return seasonal_growth_forecast(
        factor_level,
        horizons=horizons,
        growth_window=growth_window,
    ).iloc[0]


def factor_anchor_forecast(
    wide: pd.DataFrame,
    horizons,
    growth_window: int = 3,
) -> pd.DataFrame:
    factor = panel_log_factor(
        wide
    )

    last_dev = (
        np.log(
            wide.iloc[
                :,
                -1
            ]
        )
        - factor.iloc[-1]
    )

    factor_fc = factor_future(
        factor,
        horizons,
        growth_window=growth_window,
    )

    return pd.DataFrame(
        {
            h:
                np.exp(
                    last_dev
                )
                * factor_fc[h]

            for h in horizons
        },
        index=wide.index,
    )


def ets_relative_forecast(
    wide: pd.DataFrame,
    horizons,
) -> pd.DataFrame:
    horizons = list(
        horizons
    )

    factor = panel_log_factor(
        wide
    )

    relative = (
        np.log(
            wide
        )
        .sub(
            factor,
            axis=1,
        )
    )

    long = (
        relative
        .stack()
        .rename("y")
        .reset_index()
    )

    long.columns = [
        "unique_id",
        "ds",
        "y",
    ]

    sf = StatsForecast(
        models=[
            AutoETS(
                season_length=1
            )
        ],
        freq="MS",
        n_jobs=1,
    )

    fc = sf.forecast(
        df=long.dropna(),
        h=max(horizons),
    )

    if "unique_id" not in fc.columns:
        fc = fc.reset_index()

    fc["h"] = (
        fc.groupby(
            "unique_id"
        )
        .cumcount()
        + 1
    )

    rel_future = fc.pivot(
        index="unique_id",
        columns="h",
        values="AutoETS",
    )

    factor_fc = factor_future(
        factor,
        horizons,
        growth_window=1,
    )

    restored = {
        h:
            np.exp(
                rel_future[h]
            )
            * factor_fc[h]

        for h in horizons
    }

    return (
        pd.DataFrame(
            restored
        )
        .reindex(
            wide.index
        )
    )


print(
    "Classical/custom forecasting functions: READY"
)


### 3.2 Prophet — конкурсный comparator

Prophet оценивается **на том же полном multi-horizon protocol**,
что и все остальные модели.

Используется отдельная модель для каждого муниципального ряда и каждого origin.

Поскольку история короткая — от 18 до 21 месячного наблюдения на benchmark origins —
вместо высокой Fourier-размерности используется заранее зафиксированная
низкопорядковая годовая сезонность:

- linear trend;
- yearly seasonality с `Fourier order = 3`;
- weekly seasonality выключена;
- daily seasonality выключена;
- uncertainty intervals не рассчитываются.

Эта конфигурация не подбирается по Jun–Sep MAE.

Прогнозы Prophet сохраняются в отдельный cache-файл.
Если cache соответствует текущему protocol и версии конфигурации,
повторный `Run All` не требует заново обучать тысячи моделей.

In [ ]:
PROPHET_CONFIG_ID = (
    "prophet_linear_yearly_fourier3_v1"
)

PROPHET_CACHE_FILE = (
    INTERIM_DIR
    / "prophet_multihorizon_predictions_2016x4x3.parquet"
)


def prophet_cache_valid(
    df: pd.DataFrame,
) -> bool:
    required = {
        "model",
        "series_id",
        "origin",
        "h",
        "target_period",
        "prediction",
        "actual_value",
        "config_id",
    }

    if not required.issubset(
        df.columns
    ):
        return False

    if len(df) != N_FORECASTS_PER_MODEL:
        return False

    if df[
        "config_id"
    ].nunique() != 1:
        return False

    if (
        df[
            "config_id"
        ].iloc[0]
        != PROPHET_CONFIG_ID
    ):
        return False

    if df[
        [
            "model",
            "series_id",
            "origin",
            "h",
        ]
    ].duplicated().any():
        return False

    if df[
        "prediction"
    ].isna().any():
        return False

    return True


def fit_prophet_one_series(
    series_id,
    history: pd.DataFrame,
    origin: pd.Timestamp,
    horizons,
):
    train = pd.DataFrame({
        "ds":
            history.columns,

        "y":
            history.loc[
                series_id
            ].to_numpy(
                dtype=float
            ),
    })

    # Low-order yearly seasonality:
    # даём Prophet шанс моделировать сезонную форму,
    # но не используем высокую Fourier-размерность
    # на короткой истории.
    model = Prophet(
        growth="linear",
        yearly_seasonality=3,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode="additive",
        uncertainty_samples=0,
    )

    model.fit(
        train
    )

    future_dates = [
        origin
        + pd.DateOffset(
            months=int(h)
        )

        for h in horizons
    ]

    future = pd.DataFrame({
        "ds":
            future_dates
    })

    forecast = model.predict(
        future
    )

    rows = []

    for h, target, pred in zip(
        horizons,
        future_dates,
        forecast[
            "yhat"
        ].to_numpy(
            dtype=float
        ),
    ):
        rows.append({
            "model":
                "prophet",

            "series_id":
                series_id,

            "origin":
                origin,

            "h":
                int(h),

            "target_period":
                target,

            "prediction":
                float(pred),

            "actual_value":
                float(
                    wide_ref.loc[
                        series_id,
                        target,
                    ]
                ),

            "config_id":
                PROPHET_CONFIG_ID,
        })

    return rows


In [ ]:
if PROPHET_CACHE_FILE.exists():
    prophet_predictions = pd.read_parquet(
        PROPHET_CACHE_FILE
    )

    prophet_predictions[
        "origin"
    ] = pd.to_datetime(
        prophet_predictions[
            "origin"
        ]
    )

    prophet_predictions[
        "target_period"
    ] = pd.to_datetime(
        prophet_predictions[
            "target_period"
        ]
    )

    use_cache = prophet_cache_valid(
        prophet_predictions
    )

else:
    use_cache = False


if use_cache:
    print(
        "Prophet cache: VALID"
    )

else:
    print(
        "Prophet cache отсутствует или не соответствует protocol."
    )

    print(
        "Запускается полный Prophet benchmark:",
        f"{len(TEST_ORIGINS) * len(wide_ref):,}",
        "fit-операций.",
    )

    prophet_rows = []

    for origin in TEST_ORIGINS:
        history = (
            wide_ref
            .loc[
                :,
                wide_ref.columns
                <= origin
            ]
            .copy()
        )

        assert (
            history.columns.max()
            == origin
        )

        for series_id in tqdm(
            history.index,
            desc=f"Prophet {origin:%Y-%m}",
        ):
            prophet_rows.extend(
                fit_prophet_one_series(
                    series_id=series_id,
                    history=history,
                    origin=origin,
                    horizons=HORIZONS,
                )
            )

    prophet_predictions = pd.DataFrame(
        prophet_rows
    )

    assert prophet_cache_valid(
        prophet_predictions
    )

    prophet_predictions.to_parquet(
        PROPHET_CACHE_FILE,
        index=False,
    )

    print(
        "Prophet cache сохранён:",
        PROPHET_CACHE_FILE,
    )


assert prophet_cache_valid(
    prophet_predictions
)

print(
    "Prophet forecasts:",
    len(prophet_predictions),
)

print(
    "Prophet prediction range:",
    round(
        prophet_predictions[
            "prediction"
        ].min(),
        2,
    ),
    "—",
    round(
        prophet_predictions[
            "prediction"
        ].max(),
        2,
    ),
)

print(
    "Non-positive Prophet predictions:",
    int(
        (
            prophet_predictions[
                "prediction"
            ]
            <= 0
        ).sum()
    ),
)

print(
    "Prophet benchmark protocol: PASSED"
)


### 3.3 Chronos-Bolt — foundation model

В качестве современного foundation-model эксперта используется
общедоступная предобученная модель:

`amazon/chronos-bolt-base`

Chronos получает не исходный денежный уровень, а относительный log-ряд `r(i,t)`.

После прогноза относительная компонента переводится обратно
в денежную шкалу через прогноз общего фактора панели.

Так foundation model фокусируется на локальной динамике ряда,
а общая панельная сезонность моделируется отдельно.

In [ ]:
CHRONOS_DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

CHRONOS_BATCH_SIZE = (
    256
    if CHRONOS_DEVICE == "cuda"
    else 64
)

print(
    "Loading amazon/chronos-bolt-base ..."
)

print(
    "Chronos device:",
    CHRONOS_DEVICE,
)

chronos_pipeline = (
    BaseChronosPipeline
    .from_pretrained(
        "amazon/chronos-bolt-base",
        device_map=CHRONOS_DEVICE,
        torch_dtype=torch.float32,
    )
)

print(
    "Chronos pipeline:",
    type(
        chronos_pipeline
    ).__name__,
)


def chronos_relative_forecast(
    wide: pd.DataFrame,
    horizons,
    pipeline,
    batch_size: int,
) -> pd.DataFrame:
    horizons = list(
        horizons
    )

    factor = panel_log_factor(
        wide
    )

    relative = (
        np.log(
            wide
        )
        .sub(
            factor,
            axis=1,
        )
    )

    prediction_length = max(
        horizons
    )

    batches = []

    values = relative.to_numpy(
        dtype=np.float32
    )

    for start in range(
        0,
        len(relative),
        batch_size,
    ):
        block = values[
            start:
            start + batch_size
        ]

        contexts = [
            torch.tensor(
                row[
                    np.isfinite(
                        row
                    )
                ],
                dtype=torch.float32,
            )
            for row in block
        ]

        quantiles, _ = (
            pipeline
            .predict_quantiles(
                contexts,
                prediction_length=prediction_length,
                quantile_levels=[0.5],
            )
        )

        batches.append(
            quantiles[
                ...,
                0
            ]
            .detach()
            .cpu()
            .numpy()
        )

    relative_future = np.vstack(
        batches
    )

    factor_fc = factor_future(
        factor,
        horizons,
        growth_window=1,
    )

    result = {}

    for h in horizons:
        result[h] = (
            np.exp(
                relative_future[
                    :,
                    h - 1
                ]
            )
            * factor_fc[h]
        )

    return pd.DataFrame(
        result,
        index=wide.index,
    )


print(
    "Chronos relative wrapper: READY"
)


## 4. Walk-forward прогнозы остальных моделей

Каждый origin обрабатывается отдельно.

Внутрь модели передаётся только история до текущего origin включительно.

In [ ]:
def to_long_forecast(
    model_name,
    origin,
    pred,
    horizons,
):
    parts = []

    for h in horizons:
        target = (
            origin
            + pd.DateOffset(
                months=int(h)
            )
        )

        parts.append(
            pd.DataFrame({
                "model":
                    model_name,

                "series_id":
                    wide_ref.index,

                "origin":
                    origin,

                "h":
                    int(h),

                "target_period":
                    target,

                "prediction":
                    pred[
                        h
                    ].to_numpy(),

                "actual_value":
                    wide_ref[
                        target
                    ].to_numpy(),
            })
        )

    return pd.concat(
        parts,
        ignore_index=True,
    )


all_parts = []

for origin in TEST_ORIGINS:
    print(
        f"Origin: {origin:%Y-%m}"
    )

    history = (
        wide_ref
        .loc[
            :,
            wide_ref.columns
            <= origin
        ]
        .copy()
    )

    assert (
        history.columns.max()
        == origin
    )

    factor_pred = (
        factor_anchor_forecast(
            history,
            HORIZONS,
            growth_window=3,
        )
    )

    all_parts.append(
        to_long_forecast(
            "factor_anchor",
            origin,
            factor_pred,
            HORIZONS,
        )
    )

    seasonal_pred = (
        seasonal_growth_forecast(
            history,
            HORIZONS,
            growth_window=3,
        )
    )

    all_parts.append(
        to_long_forecast(
            "seasonal_growth",
            origin,
            seasonal_pred,
            HORIZONS,
        )
    )

    ets_pred = (
        ets_relative_forecast(
            history,
            HORIZONS,
        )
    )

    all_parts.append(
        to_long_forecast(
            "ets_relative",
            origin,
            ets_pred,
            HORIZONS,
        )
    )

    chronos_pred = (
        chronos_relative_forecast(
            history,
            HORIZONS,
            pipeline=chronos_pipeline,
            batch_size=CHRONOS_BATCH_SIZE,
        )
    )

    all_parts.append(
        to_long_forecast(
            "chronos_relative",
            origin,
            chronos_pred,
            HORIZONS,
        )
    )


baseline_predictions = pd.concat(
    all_parts,
    ignore_index=True,
)

print(
    "Rows:",
    len(
        baseline_predictions
    ),
)

display(
    baseline_predictions
    .groupby(
        "model"
    )
    .size()
    .rename(
        "n_predictions"
    )
    .to_frame()
)

assert (
    len(
        baseline_predictions
    )
    == N_FORECASTS_PER_MODEL * 4
)

assert not baseline_predictions.duplicated(
    [
        "model",
        "series_id",
        "origin",
        "h",
    ]
).any()

assert (
    baseline_predictions
    .isna()
    .sum()
    .sum()
    == 0
)

print(
    "Walk-forward forecasts: PASSED"
)


## 5. Reference forecasting ensemble

Reference ensemble строится как простое среднее трёх
методологически различных экспертов:

- Chronos-Bolt relative;
- AutoETS relative;
- Seasonal Growth.

Вес каждого эксперта равен `1/3`.

Веса **не подбираются на оцениваемом Jun–Sep периоде**.

Prophet используется как независимый comparator
и не входит в ensemble.

In [ ]:
ENSEMBLE_MEMBERS = [
    "chronos_relative",
    "ets_relative",
    "seasonal_growth",
]

wide_pred = (
    baseline_predictions
    .pivot(
        index=[
            "series_id",
            "origin",
            "h",
            "target_period",
            "actual_value",
        ],
        columns="model",
        values="prediction",
    )
    .reset_index()
)

wide_pred.columns.name = None

wide_pred[
    "ensemble_mean"
] = (
    wide_pred[
        ENSEMBLE_MEMBERS
    ]
    .mean(
        axis=1
    )
)

wide_pred[
    "ensemble_abs_error"
] = (
    wide_pred[
        "actual_value"
    ]
    - wide_pred[
        "ensemble_mean"
    ]
).abs()

assert len(
    wide_pred
) == N_FORECASTS_PER_MODEL

assert not wide_pred.duplicated(
    [
        "series_id",
        "origin",
        "h",
    ]
).any()

print(
    "Ensemble rows:",
    len(
        wide_pred
    ),
)

print(
    "Reference ensemble MAE:",
    round(
        wide_pred[
            "ensemble_abs_error"
        ].mean(),
        3,
    ),
)


## 6. Итоговое сравнение моделей

Все модели ниже оцениваются на одинаковых 24 192 forecast observations.

Главная метрика — **MAE**.

`R²` приводится дополнительно и не используется для выбора победителя.

In [ ]:
def r2_score_simple(
    actual,
    prediction,
):
    actual = np.asarray(
        actual,
        dtype=float,
    )

    prediction = np.asarray(
        prediction,
        dtype=float,
    )

    ss_res = np.sum(
        (
            actual
            - prediction
        ) ** 2
    )

    ss_tot = np.sum(
        (
            actual
            - actual.mean()
        ) ** 2
    )

    return (
        1.0
        - ss_res
        / ss_tot
    )


benchmark_predictions = pd.concat(
    [
        baseline_predictions,
        prophet_predictions[
            [
                "model",
                "series_id",
                "origin",
                "h",
                "target_period",
                "prediction",
                "actual_value",
            ]
        ],
    ],
    ignore_index=True,
)

summary_rows = []

for model, g in (
    benchmark_predictions
    .groupby(
        "model"
    )
):
    err = (
        g[
            "actual_value"
        ]
        - g[
            "prediction"
        ]
    ).abs()

    h_values = g[
        "h"
    ].to_numpy()

    summary_rows.append({
        "model":
            model,

        "MAE":
            float(
                err.mean()
            ),

        "R2":
            float(
                r2_score_simple(
                    g[
                        "actual_value"
                    ],
                    g[
                        "prediction"
                    ],
                )
            ),

        "MAE_h1":
            float(
                err[
                    h_values
                    == 1
                ].mean()
            ),

        "MAE_h2":
            float(
                err[
                    h_values
                    == 2
                ].mean()
            ),

        "MAE_h3":
            float(
                err[
                    h_values
                    == 3
                ].mean()
            ),

        "n_predictions":
            int(
                len(g)
            ),
    })


summary_rows.append({
    "model":
        "ensemble_mean",

    "MAE":
        float(
            wide_pred[
                "ensemble_abs_error"
            ].mean()
        ),

    "R2":
        float(
            r2_score_simple(
                wide_pred[
                    "actual_value"
                ],
                wide_pred[
                    "ensemble_mean"
                ],
            )
        ),

    "MAE_h1":
        float(
            wide_pred.loc[
                wide_pred["h"]
                == 1,
                "ensemble_abs_error",
            ].mean()
        ),

    "MAE_h2":
        float(
            wide_pred.loc[
                wide_pred["h"]
                == 2,
                "ensemble_abs_error",
            ].mean()
        ),

    "MAE_h3":
        float(
            wide_pred.loc[
                wide_pred["h"]
                == 3,
                "ensemble_abs_error",
            ].mean()
        ),

    "n_predictions":
        int(
            len(
                wide_pred
            )
        ),
})


benchmark_summary = (
    pd.DataFrame(
        summary_rows
    )
    .sort_values(
        "MAE"
    )
    .reset_index(
        drop=True
    )
)

display(
    benchmark_summary
    .round(4)
)


assert (
    benchmark_summary[
        "n_predictions"
    ]
    .eq(
        N_FORECASTS_PER_MODEL
    )
    .all()
)

print(
    "Unified benchmark table: PASSED"
)


In [ ]:
DISPLAY_NAMES = {
    "factor_anchor":
        "Factor Anchor",

    "seasonal_growth":
        "Seasonal Growth",

    "ets_relative":
        "AutoETS Relative",

    "chronos_relative":
        "Chronos-Bolt Relative",

    "prophet":
        "Prophet",

    "ensemble_mean":
        "Reference Ensemble",
}

plot_table = (
    benchmark_summary[
        [
            "model",
            "MAE",
        ]
    ]
    .assign(
        display_name=lambda x:
            x[
                "model"
            ]
            .map(
                DISPLAY_NAMES
            )
    )
    .set_index(
        "display_name"
    )
)

ax = (
    plot_table[
        "MAE"
    ]
    .sort_values(
        ascending=False
    )
    .plot(
        kind="barh",
        figsize=(9, 5),
    )
)

ax.set_title(
    "Forecasting benchmark: MAE"
)

ax.set_xlabel(
    "MAE"
)

ax.set_ylabel(
    ""
)

plt.tight_layout()
plt.show()


## 7. Прямое сравнение с Prophet

Этот блок отвечает на конкурсный вопрос:
**способна ли выбранная forecasting-система превзойти Prophet
на одном и том же evaluation protocol?**

Результат вычисляется непосредственно из общей benchmark-таблицы.

In [ ]:
prophet_mae = float(
    benchmark_summary.loc[
        benchmark_summary[
            "model"
        ]
        == "prophet",
        "MAE",
    ].iloc[0]
)

ensemble_mae = float(
    benchmark_summary.loc[
        benchmark_summary[
            "model"
        ]
        == "ensemble_mean",
        "MAE",
    ].iloc[0]
)

gain_vs_prophet = (
    prophet_mae
    - ensemble_mae
)

gain_vs_prophet_pct = (
    100.0
    * gain_vs_prophet
    / prophet_mae
)

prophet_comparison = pd.DataFrame([
    {
        "model":
            "Prophet",

        "MAE":
            prophet_mae,

        "gain_vs_prophet":
            0.0,

        "gain_vs_prophet_pct":
            0.0,
    },
    {
        "model":
            "Reference forecasting ensemble",

        "MAE":
            ensemble_mae,

        "gain_vs_prophet":
            gain_vs_prophet,

        "gain_vs_prophet_pct":
            gain_vs_prophet_pct,
    },
])

display(
    prophet_comparison
    .round(3)
)


if gain_vs_prophet > 0:
    print(
        "Reference ensemble превосходит Prophet:",
        f"{gain_vs_prophet:.3f} MAE",
        f"({gain_vs_prophet_pct:.2f}%).",
    )

else:
    print(
        "На этом protocol Prophet оказался не хуже reference ensemble."
    )

print(
    "Prophet comparison uses exactly the same 24,192 forecast cells: PASSED"
)


## 8. Диагностика Reference Ensemble

Смотрим стабильность итогового reference ensemble
по forecast origin и горизонту.

In [ ]:
ensemble_by_origin = (
    wide_pred
    .groupby(
        "origin"
    )[
        "ensemble_abs_error"
    ]
    .mean()
    .rename(
        "MAE"
    )
    .reset_index()
)

ensemble_by_horizon = (
    wide_pred
    .groupby(
        "h"
    )[
        "ensemble_abs_error"
    ]
    .mean()
    .rename(
        "MAE"
    )
    .reset_index()
)

print(
    "Reference ensemble by origin:"
)

display(
    ensemble_by_origin
    .round(3)
)

print(
    "Reference ensemble by horizon:"
)

display(
    ensemble_by_horizon
    .round(3)
)


## 9. Контроль воспроизводимости

Для собственных deterministic/statistical компонентов
и frozen Chronos setup фиксируются контрольные MAE.

Prophet проверяется по структуре protocol и полноте прогнозов,
но его точное значение MAE не зашивается в `assert`,
поскольку оно может незначительно зависеть от версии Prophet/CmdStan.

Это позволяет обнаруживать реальные изменения pipeline,
не превращая version-sensitive comparator в хрупкий regression test.

In [ ]:
EXPECTED = {
    "factor_anchor":
        939.265,

    "chronos_relative":
        885.763,

    "ets_relative":
        925.792,

    "seasonal_growth":
        995.026,

    "ensemble_mean":
        761.916,
}

observed = dict(
    zip(
        benchmark_summary[
            "model"
        ],
        benchmark_summary[
            "MAE"
        ],
    )
)

for model, expected in (
    EXPECTED.items()
):
    actual = float(
        observed[
            model
        ]
    )

    print(
        f"{model:18s}: "
        f"actual={actual:.3f}, "
        f"expected≈{expected:.3f}"
    )

    assert np.isclose(
        actual,
        expected,
        atol=0.10,
    ), (
        model,
        actual,
        expected,
    )


EXPECTED_H = {
    1: 660.931,
    2: 736.020,
    3: 888.797,
}

observed_h = dict(
    zip(
        ensemble_by_horizon[
            "h"
        ],
        ensemble_by_horizon[
            "MAE"
        ],
    )
)

for h, expected in (
    EXPECTED_H.items()
):
    assert np.isclose(
        observed_h[h],
        expected,
        atol=0.10,
    )


EXPECTED_ORIGIN = {
    pd.Timestamp(
        "2024-06-01"
    ): 869.612,

    pd.Timestamp(
        "2024-07-01"
    ): 673.852,

    pd.Timestamp(
        "2024-08-01"
    ): 740.102,

    pd.Timestamp(
        "2024-09-01"
    ): 764.099,
}

observed_origin = dict(
    zip(
        pd.to_datetime(
            ensemble_by_origin[
                "origin"
            ]
        ),
        ensemble_by_origin[
            "MAE"
        ],
    )
)

for origin, expected in (
    EXPECTED_ORIGIN.items()
):
    assert np.isclose(
        observed_origin[
            origin
        ],
        expected,
        atol=0.10,
    )


assert prophet_cache_valid(
    prophet_predictions
)

assert np.isfinite(
    prophet_mae
)

print(
    "Forecasting benchmark reproducibility: PASSED"
)


## 10. Сохранение артефактов

Основные downstream-файлы предыдущей архитектуры сохраняются
без изменения их назначения.

Prophet сохраняется отдельно, чтобы не менять контракт
`baseline_multihorizon_predictions.parquet`, который используется
следующими notebooks.

In [ ]:
PRED_FILE = (
    INTERIM_DIR
    / "baseline_multihorizon_predictions.parquet"
)

ENSEMBLE_FILE = (
    INTERIM_DIR
    / "baseline_ensemble_predictions.parquet"
)

SUMMARY_FILE = (
    INTERIM_DIR
    / "baseline_multihorizon_summary.csv"
)

PROPHET_FILE = (
    INTERIM_DIR
    / "prophet_multihorizon_predictions_2016x4x3.parquet"
)

PROPHET_COMPARISON_FILE = (
    INTERIM_DIR
    / "prophet_comparison_summary.csv"
)

MANIFEST_FILE = (
    INTERIM_DIR
    / "baseline_multihorizon_manifest.json"
)


baseline_predictions.to_parquet(
    PRED_FILE,
    index=False,
)

wide_pred.to_parquet(
    ENSEMBLE_FILE,
    index=False,
)

benchmark_summary.to_csv(
    SUMMARY_FILE,
    index=False,
)

prophet_predictions.to_parquet(
    PROPHET_FILE,
    index=False,
)

prophet_comparison.to_csv(
    PROPHET_COMPARISON_FILE,
    index=False,
)


manifest = {
    "name":
        "Final forecasting model benchmark",

    "primary_metric":
        "MAE",

    "secondary_metric":
        "R2",

    "data_panel":
        "2016 complete municipal series, Jan-2023 to Dec-2024",

    "source_sha256":
        data_manifest[
            "source"
        ][
            "sha256"
        ],

    "origins":
        [
            d.strftime(
                "%Y-%m-%d"
            )
            for d in TEST_ORIGINS
        ],

    "horizons":
        HORIZONS,

    "n_series":
        int(
            N_SERIES
        ),

    "n_forecasts_per_model":
        int(
            N_FORECASTS_PER_MODEL
        ),

    "methods": {
        "prophet": {
            "config_id":
                PROPHET_CONFIG_ID,

            "growth":
                "linear",

            "yearly_fourier_order":
                3,

            "weekly_seasonality":
                False,

            "daily_seasonality":
                False,

            "uncertainty_samples":
                0,
        },

        "factor_anchor":
            "custom panel-factor baseline",

        "seasonal_growth": {
            "growth_window":
                3,
        },

        "ets_relative":
            "AutoETS(season_length=1) on relative log-level",

        "chronos_relative": {
            "checkpoint":
                "amazon/chronos-bolt-base",

            "quantile":
                0.5,

            "representation":
                "relative log-level",
        },

        "ensemble_mean": {
            "members":
                ENSEMBLE_MEMBERS,

            "weights":
                [
                    1 / 3,
                    1 / 3,
                    1 / 3,
                ],

            "tuned_on_test":
                False,
        },
    },

    "mae": {
        row[
            "model"
        ]:
            float(
                row[
                    "MAE"
                ]
            )

        for _, row in (
            benchmark_summary
            .iterrows()
        )
    },

    "r2": {
        row[
            "model"
        ]:
            float(
                row[
                    "R2"
                ]
            )

        for _, row in (
            benchmark_summary
            .iterrows()
        )
    },

    "prophet_comparison": {
        "prophet_mae":
            prophet_mae,

        "reference_ensemble_mae":
            ensemble_mae,

        "gain_vs_prophet":
            gain_vs_prophet,

        "gain_vs_prophet_pct":
            gain_vs_prophet_pct,
    },

    "methodology":
        (
            "All models are evaluated on identical time-safe forecast cells. "
            "Prophet is a separate comparator and is not part of the ensemble. "
            "Jun-Sep is a development/challenger benchmark, not a new untouched holdout."
        ),
}


with MANIFEST_FILE.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2,
    )


artifact_table = pd.DataFrame([
    {
        "artifact":
            PRED_FILE.name,

        "exists":
            PRED_FILE.exists(),
    },
    {
        "artifact":
            ENSEMBLE_FILE.name,

        "exists":
            ENSEMBLE_FILE.exists(),
    },
    {
        "artifact":
            SUMMARY_FILE.name,

        "exists":
            SUMMARY_FILE.exists(),
    },
    {
        "artifact":
            PROPHET_FILE.name,

        "exists":
            PROPHET_FILE.exists(),
    },
    {
        "artifact":
            PROPHET_COMPARISON_FILE.name,

        "exists":
            PROPHET_COMPARISON_FILE.exists(),
    },
    {
        "artifact":
            MANIFEST_FILE.name,

        "exists":
            MANIFEST_FILE.exists(),
    },
])

display(
    artifact_table
)

assert artifact_table[
    "exists"
].all()

print(
    "Artifacts saved to:",
    INTERIM_DIR,
)

print(
    "FINAL FORECASTING BASELINES: PASSED"
)


## 11. Итог

На едином multi-horizon protocol выполнено прямое сравнение:

- Prophet;
- Seasonal Growth;
- Factor Anchor;
- AutoETS;
- Chronos-Bolt-base;
- Reference forecasting ensemble.

Для каждой модели используется ровно **24 192 forecast observations**.

Контрольное значение Reference forecasting ensemble:

- общий MAE — **≈ 761.916**;
- `h1` — **≈ 660.931**;
- `h2` — **≈ 736.020**;
- `h3` — **≈ 888.797**.

Prophet теперь оценивается на **том же protocol**, поэтому сравнение
`Reference Ensemble vs Prophet` является прямым и методологически корректным.

Точный Prophet MAE сохраняется в:

- `baseline_multihorizon_summary.csv`;
- `prophet_comparison_summary.csv`;
- `baseline_multihorizon_manifest.json`.

### Что доказано этим этапом

1. Конкурсная MAE используется как основная метрика.
2. Prophet включён в единый честный benchmark.
3. Классическая статистическая модель AutoETS включена.
4. Современная foundation model Chronos-Bolt включена.
5. Итоговый reference ensemble не подбирает веса на Jun–Sep evaluation period.
6. Все forecasting-модели соблюдают временную причинность.

Следующий notebook строит собственную статическую forecasting-архитектуру
и сравнивает её с этим frozen reference.